# Drive and utilities


In [4]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [5]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle

def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)


# Initial cleaning and combining

In [1]:
#@title Load citations data and acquired patents
#!pip install pandas
#!pip install numpy
import pandas as pd
import numpy as np

# Paths
path_pat = "/content/drive/MyDrive/PhD Data/09 Acquired patents/04 All patents.dta"
patents = pd.read_stata(path_pat)

In [2]:
#@title Clean acquired patents

## Keep acquired patents only
patents = patents[patents['acquired'] == 1]

## Drop patents that are resold based on resold_date
patents = patents[patents['resold_date'].isna()]

## Drop empty CPCs - these are mostly Design patents or patents with no CPC in the database
patents = patents[patents['cpc_subclass_current'].notna()]

# Drop desing patents - they have no abstracts & reissue patents
patents = patents[patents['patent_type'] == "utility"]

## Keep relevant vars
patents = patents[['patent_id', 'acq_date', 'acq_year', 'grant_date', 'cpc_subclass_current']]

## Get grant year
patents["grant_date"] = pd.to_datetime(patents["grant_date"])
patents["grant_year"] = patents["grant_date"].dt.year
patents.rename(columns={"cpc_subclass_current": "cpc_subclass"}, inplace=True)



In [6]:
#@title Load BERT embeddings (float16) and combine with CPC data

# Load cpc Data for all patents
embeddings = load("/content/drive/MyDrive/PhD Data/01 CLS Embeddings/All embeddings - float16/all_embeddings_float16.pkl")

# Load processed Patents dataset containing IDs and CPCs
pat_processed = load("/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Patents with tech fields.pkl")

# Convert embeddings dictionary into a pandas Series (the keys will become the index)
embedding_series = pd.Series(embeddings, name='embedding')

# Join on the index (which now is patent_id)
pat_processed = pat_processed.join(embedding_series)

# Optionally, drop rows where no embedding was found.
pat_processed = pat_processed.dropna(subset=['embedding'])


In [7]:
pat_processed.head(10)

,cpc_subclass,application_year,grant_year,embedding
patent_id,,,,
10000000,G01S,2015,2018,"[-0.2319, -0.5854, -0.3345, 0.1771, 0.462, -0...."
10000001,B29C,2015,2018,"[0.586, -1.353, -0.06964, 0.3408, -0.3367, -0...."
10000002,B29C,2014,2018,"[-0.004852, 0.593, -1.2705, 0.2257, -0.796, 0...."
10000003,B29C,2013,2018,"[-0.257, -0.998, -0.62, 0.8584, 0.4792, -0.139..."
10000004,B29C,2015,2018,"[-0.6353, -1.116, -0.4683, -0.1401, 0.03093, -..."
10000005,B29C,2012,2018,"[-0.4216, -0.11597, -0.6816, 0.6265, 0.1705, -..."
10000006,B29C,2015,2018,"[0.2878, -1.322, -1.132, 1.071, -0.532, -0.923..."
10000007,B29C,2016,2018,"[-0.8057, -0.9014, 0.5444, -0.0222, 0.178, 0.9..."
10000008,B29C,2014,2018,"[-1.5625, -0.4666, -1.394, 1.299, -1.194, 0.38..."


In [8]:
#@title Combine it with acquired patents
pat_processed.reset_index(inplace=True)

acquired_patents = patents.merge(pat_processed, on='patent_id', how="left")
acquired_patents = acquired_patents.dropna(subset=['embedding']) ## drops 71 patents with no BERT embeddings
assert len(acquired_patents) == 19649

## Drop duplicate columns
if acquired_patents['cpc_subclass_x'].equals(acquired_patents['cpc_subclass_y']):
    acquired_patents.drop('cpc_subclass_y', axis=1, inplace=True)
if acquired_patents['grant_year_x'].astype(float).equals(acquired_patents['grant_year_y'].astype(float)):
    acquired_patents.drop('grant_year_y', axis=1, inplace=True)

acquired_patents.rename(columns={"cpc_subclass_x": "cpc_subclass", "grant_year_x": "grant_year"}, inplace=True)

# Get unique CPC combinations in acquired patents
acquired_cpc = acquired_patents[['cpc_subclass']].drop_duplicates()

# Filter the embeddings DataFrame to keep only relevant CPC-Year patents
potential_controls = pat_processed.merge(acquired_cpc, on=['cpc_subclass'], how='inner')

# Check the filtered dataset
print(potential_controls.head())
print(f"Remaining patents with the same CPCs as treated ones: {len(potential_controls)}, before it was 2.896.496")


  patent_id cpc_subclass  application_year  grant_year  \
0  10000000         G01S              2015        2018   
1  10000001         B29C              2015        2018   
2  10000002         B29C              2014        2018   
3  10000003         B29C              2013        2018   
4  10000004         B29C              2015        2018   

                                           embedding  
0  [-0.2319, -0.5854, -0.3345, 0.1771, 0.462, -0....  
1  [0.586, -1.353, -0.06964, 0.3408, -0.3367, -0....  
2  [-0.004852, 0.593, -1.2705, 0.2257, -0.796, 0....  
3  [-0.257, -0.998, -0.62, 0.8584, 0.4792, -0.139...  
4  [-0.6353, -1.116, -0.4683, -0.1401, 0.03093, -...  
Remaining patents with the same CPCs as treated ones: 5960763, before it was 2.896.496


In [9]:
#@title Check that every acquired patent is in the list of potential controls and remove them

# Get the acquired IDs
acquired_patent_ids = set(acquired_patents["patent_id"].astype(str))

# Get the potential control IDs
potential_controls = potential_controls.reset_index()
potential_controls_ids = set(potential_controls['patent_id'].astype(str))

# Check if acquired_patent_ids is a subset of potential_controls_ids
all_acquired_in_controls = acquired_patent_ids.issubset(potential_controls_ids)
assert all_acquired_in_controls == True

# Remove
assert len(potential_controls) == 5960763
potential_controls = potential_controls[~potential_controls['patent_id'].isin(acquired_patent_ids)]
assert len(potential_controls) == 5960763 - 19649

# Remove from potential controls for which the abstract is the same


## Checks
cols = ['patent_id', 'cpc_subclass', 'application_year', 'grant_year', 'embedding']
# Check if all values in specified columns are not missing
no_missing_values = potential_controls[cols].notna().all().all()

if no_missing_values:
    print("No missing values in the specified columns.")
else:
    print("There are missing values in the specified columns.")

potential_controls = potential_controls[cols]

cols = ['patent_id', 'cpc_subclass', 'application_year', 'grant_year', 'grant_date', 'acq_date', 'acq_year',  'embedding']
no_missing_values = acquired_patents[cols].notna().all().all()

if no_missing_values:
    print("No missing values in the specified columns.")
else:
    print("There are missing values in the specified columns.")

acquired_patents = acquired_patents[cols]


No missing values in the specified columns.
No missing values in the specified columns.


In [29]:
#@title Save

#acquired_patents.to_csv("/content/drive/MyDrive/PhD Data/10 Sample - pre final/acquired_patents_noresold.csv", index=False)
acquired_patents.to_pickle("/content/drive/MyDrive/PhD Data/10 Sample - pre final/acquired_patents_noresold.pkl")
## Save potential controls as CSV with no embedding columns
#potenttial_controls_no_emb = potential_controls.drop('embedding', axis=1)
#potenttial_controls_no_emb.to_csv("/content/drive/MyDrive/PhD Data/10 Sample - pre final/potential_controls_no_exact_match_grantyear_cpc.csv", index=False)


In [22]:
del globals()['embedding_series']
del globals()['patents']
del globals()['pat_processed']

In [23]:
file_path = '/content/drive/MyDrive/PhD Data/10 Sample - pre final/potential_controls_no_exact_match_grantyear_cpc.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(potential_controls, f)

In [24]:
## Save potential controls as CSV with no embedding columns

potenttial_controls_no_emb = potential_controls.drop('embedding', axis=1)
potenttial_controls_no_emb.to_csv("/content/drive/MyDrive/PhD Data/10 Sample - pre final/potential_controls_no_exact_match_grantyear_cpc.csv", index=False)

In [25]:
np.arange(0, 1 + 0.25, 0.25)


array([0.  , 0.25, 0.5 , 0.75, 1.  ])